# KeyPrint: Recognising People by How They Type

**Course:** UML501 Machine Learning Lab · **Dataset:** CMU Keystroke Dynamics Benchmark (Killourhy & Maxion, 2009)

This notebook is the full study behind the KeyPrint app. It is written for someone who has **never studied machine learning**: every step says *what* we do, *why*, and *what to look for* in the output.

**How to use it:** run the cells from top to bottom (`Shift + Enter`). Read the text between them; it is your explanation for the viva.

| Part | What happens | PDF deliverable |
|---|---|---|
| 1 | The problem and why it matters | Problem definition |
| 2 | Load and describe the data | Dataset documentation |
| 3 | Look at the data with graphs | EDA |
| 4 | Split and scale the data | Preprocessing |
| 5 | Baseline model | Baseline |
| 6–9 | Experiments with KNN, logistic regression, PCA | Proposed method + experiments |
| 10 | Final test, confusion matrix | Evaluation |
| 11 | Login check: FAR, FRR, EER | Evaluation + trade-offs |
| 12 | Optional: test on your group's own typing | Extra |
| 13 | Conclusions (you fill these in) | Analysis and conclusion |

## 1 · The problem

A password only checks **what** you type. If someone steals it, they get in.

But every person types with a personal **rhythm**: how long they hold each key, and how long they pause between keys. That rhythm is hard to copy, even for someone who knows the password.

**Research questions**
1. Can a machine learning model tell **who** is typing, just from timing? *(identification)*
2. Can it decide whether the person is **who they claim to be**? *(verification, i.e. a login)*
3. Which settings matter: scaling, the number of neighbours K, PCA size, amount of training data?

**Background, in one paragraph:** this idea is called *keystroke dynamics*, a kind of *behavioural biometric* (like a signature, unlike a fingerprint). Killourhy & Maxion (2009) collected this dataset and compared 14 detectors; their best ones made about 1 mistake in 10 at the equal-error point. We use their data, reproduce a simple version of their best detector (scaled Manhattan distance), and add classifiers from our course: KNN and logistic regression, plus PCA.

**What a "feature" is here** (example with the tiny password `cat` + Enter):

| key | pressed at (ms) | released at (ms) |
|---|---|---|
| c | 0 | 100 |
| a | 250 | 330 |
| t | 500 | 590 |
| Enter | 800 | 880 |

* hold time of `c` = 100 − 0 = **100 ms** (H)
* press→press `c→a` = 250 − 0 = **250 ms** (DD)
* release→press `c→a` = 250 − 100 = **150 ms** (UD)

4 keys give 4 + 3 + 3 = 10 numbers. The real password `.tie5Roanl` + Enter has 11 keys, so **11 + 10 + 10 = 31 numbers** per attempt.

## 2 · Setup and loading the data

First we import the tools.

* **pandas**: tables (like Excel inside Python)
* **NumPy**: fast maths on lists of numbers
* **scikit-learn**: the machine learning models
* **src/**: our own small helper files in this project (open them; they are commented line by line)

In [ ]:
import sys, platform
from pathlib import Path

# Let Python find our own code in the src/ folder (one level up from notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt

from src.config import CMU_CSV, FEATURE_COLUMNS, RANDOM_STATE, TABLES_DIR
from src.data import load_cmu, split_by_session, X_y
from src import experiments as ex
from src import plots

np.random.seed(RANDOM_STATE)   # same random numbers every run -> reproducible
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Python      ", platform.python_version())
print("pandas      ", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("Data file   ", CMU_CSV)

In [ ]:
df = load_cmu(CMU_CSV)      # reads data/DSL-StrongPasswordData.csv
df.head()                   # show the first 5 rows

**What you are looking at:** each row is **one attempt** at typing the password.

* `subject` is **who** typed it (the *label*, the answer we want the model to predict)
* `sessionIndex` is which sitting (1 to 8, on different days)
* `rep` is the attempt number inside that sitting (1 to 50)
* the other **31 columns** are the timings in **seconds** (the *features*)

In [ ]:
print("Rows (attempts):        ", len(df))
print("People (subjects):      ", df["subject"].nunique())
print("Sessions per person:    ", sorted(df["sessionIndex"].unique().tolist()))
print("Attempts per session:   ", df.groupby(["subject", "sessionIndex"]).size().unique().tolist())
print("Features:               ", len(FEATURE_COLUMNS))
print("Missing values in total:", int(df[FEATURE_COLUMNS].isna().sum().sum()))

### Dataset documentation

| Item | Value |
|---|---|
| Name | CMU Keystroke Dynamics Benchmark Data Set (`DSL-StrongPasswordData.csv`) |
| Source | https://www.cs.cmu.edu/~keystroke/ |
| Paper | K. S. Killourhy and R. A. Maxion, "Comparing Anomaly-Detection Algorithms for Keystroke Dynamics", DSN 2009 |
| Usage | Shared by the authors for research use; cite the paper above |
| Samples | 20,400 attempts = 51 people × 8 sessions × 50 attempts |
| Features | 31 timing features (11 H, 10 DD, 10 UD), in seconds |
| Target | `subject` (51 classes) |
| Limitations | One fixed password for everyone; collected in a lab on one keyboard; volunteers typed it 400 times, so they got very practised; subject IDs are not continuous (e.g. there is no s001) |

In [ ]:
# Quick numeric summary of a few features (in seconds)
df[["H.period", "DD.period.t", "UD.period.t", "H.Return"]].describe().round(4)

👉 **What to notice:** `UD` (release → press) can be **negative**. That happens when you press the next key *before* letting go of the previous one, which fast typists do all the time. It is not an error in the data, so we keep it.

## 3 · Exploratory Data Analysis (EDA)

EDA means *looking at the data before modelling*, to understand it and to spot problems.

In [ ]:
plots.eda_samples_per_subject(df);

**EDA 1 · Class balance.** Every person has exactly 400 attempts, so the classes are **perfectly balanced**. We don't need any special treatment for imbalance, and accuracy is a fair metric (we still report macro-F1 too).

In [ ]:
plots.eda_feature_distributions(df);

**EDA 2 · Distributions.** Hold times are short and tightly grouped. Gaps (DD, UD) are bigger and spread out more, with a long tail of slow attempts to the right. Because gaps are bigger numbers, they would dominate any distance calculation, which is exactly why we **scale** the data in Part 4.

In [ ]:
plots.eda_correlation(df);

**EDA 3 · Correlations.** Red squares mean two features rise and fall together. Look for strong pairs like `DD.x.y` with `UD.x.y`: they are linked by definition (DD = H + UD). Strongly related features carry overlapping information, which hints that **PCA** can compress the 31 features without losing much (Part 8).

In [ ]:
plots.eda_subject_rhythms(df);

**EDA 4 · Everyone has a rhythm.** Each line is one person's average for all 31 features; the shaded band is how much they vary. The lines have different shapes, and that difference is the "typing fingerprint" the models learn.

In [ ]:
plots.eda_pca_preview(df);

**EDA 5 · A first PCA map.** PCA squeezes 31 numbers into 2 so we can draw them. Each dot is one attempt, coloured by person. Even in only 2 dimensions, many people form their **own cluster**. Where clusters overlap, those people type alike and the models will find them harder to separate.

## 4 · Preprocessing

### 4.1 Splitting: train, validation, test

We never test a model on attempts it learned from. That would be like getting the exam questions to study from.

| Part | Sessions | Used for |
|---|---|---|
| **Train** | 1–5 | the model learns from these |
| **Validation** | 6 | "practice exam": we choose settings like K here |
| **Test** | 7–8 | "final exam": used **once**, at the very end |

We split **by session**, not randomly. Attempts from the same sitting are almost identical, so a random split would put near-copies in both train and test and make results look better than they really are. Splitting by session answers the honest question: *can we recognise you on a later day?*

In [ ]:
train, val, test = split_by_session(df)
trainval = pd.concat([train, val])     # used for the final model (train + validation)

print(f"Train      {len(train):>6} rows  sessions {sorted(train.sessionIndex.unique().tolist())}")
print(f"Validation {len(val):>6} rows  sessions {sorted(val.sessionIndex.unique().tolist())}")
print(f"Test       {len(test):>6} rows  sessions {sorted(test.sessionIndex.unique().tolist())}")

X_train, y_train = X_y(train)
X_val, y_val = X_y(val)

### 4.2 Scaling (z-score)

For each feature: **new value = (value − mean) ÷ standard deviation**. After scaling, 0 means "average", +2 means "much higher than usual", −1 means "a bit lower than usual". Every feature now counts equally.

⚠️ The mean and standard deviation are learned from the **training data only**, then applied to validation and test. If we used the test data to compute them, information from the "final exam" would leak into training (*data leakage*). In the experiments, scikit-learn's `Pipeline` does this for us automatically.

Other steps, and why they are **not** needed here:
* **Missing values:** there are none (checked above).
* **Encoding categories:** all features are already numbers; the label is just a name.
* **Imbalance handling:** classes are perfectly balanced (EDA 1).
* **Outliers:** we keep slow attempts; real users also type slowly sometimes.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)          # learn mean and std from TRAIN only
X_train_scaled = scaler.transform(X_train)

demo = pd.DataFrame({
    "feature": FEATURE_COLUMNS[:4],
    "mean before (s)": X_train[:, :4].mean(axis=0).round(4),
    "std before (s)": X_train[:, :4].std(axis=0).round(4),
    "mean after": X_train_scaled[:, :4].mean(axis=0).round(2),
    "std after": X_train_scaled[:, :4].std(axis=0).round(2),
})
demo

After scaling every feature has mean 0 and standard deviation 1.

## 5 · Baseline: the simplest reasonable model

A **baseline** is a simple starting point. Later models must beat it to be worth anything.

Our baseline is **KNN with K = 1 and no scaling**: for a new attempt, find the single most similar stored attempt and copy its name.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score

baseline = KNeighborsClassifier(n_neighbors=1)
baseline.fit(X_train, y_train)          # KNN "training" = just remembering every attempt
pred = baseline.predict(X_val)          # for each validation attempt, find its nearest neighbour

print("Baseline validation accuracy:", round(accuracy_score(y_val, pred) * 100, 2), "%")
print("Baseline validation macro-F1:", round(f1_score(y_val, pred, average='macro') * 100, 2), "%")
print("Random guessing would get about", round(100 / df['subject'].nunique(), 2), "%")

## 6 · Experiment 1: K and scaling

**KNN in plain words:** "Tell me who your neighbours are and I'll tell you who you are."
1. Measure the distance from the new attempt to every stored attempt.
2. Take the **K** closest.
3. Majority vote on their names.

We try several K, each **with and without scaling**, and score them on the **validation** set.

In [ ]:
k_table = ex.exp_k_and_scaling(train, val)
k_table.to_csv(TABLES_DIR / "exp1_k_and_scaling.csv", index=False)
best_k = ex.best_k(k_table)
print("Best K (scaled, by validation macro-F1):", best_k)
k_table.round(4)

In [ ]:
plots.plot_k_scaling(k_table, best_k);

👉 **What to notice:**
* Does the scaled line sit above the unscaled one? If so, scaling helped, because without it the large gap features drowned out the hold times.
* K = 1 trusts a single neighbour, which can be noisy. Very large K pulls in other people's attempts. The best K is usually somewhere in between.

*Write your own observation here in one or two sentences.*

## 7 · Logistic regression

**In plain words:** the model gives each of the 31 features a **weight** for each person. For a new attempt it multiplies and adds, then turns the scores into **probabilities** that add up to 100% (with 51 people this is called *multinomial* logistic regression, using the softmax function).

**Training** = adjusting the weights again and again so the probabilities on training data get closer to the right answers.

In [ ]:
lr_scaled, lr_pred = ex.fit_and_score(ex.make_logreg(scaled=True), X_train, y_train, X_val, y_val)
lr_raw, _ = ex.fit_and_score(ex.make_logreg(scaled=False), X_train, y_train, X_val, y_val)
pd.DataFrame([{"model": "Logistic regression, no scaling", **lr_raw},
              {"model": "Logistic regression, scaled", **lr_scaled}]).round(4)

Logistic regression also tells us **how confident** it is. Here are its top 3 guesses for one validation attempt:

In [ ]:
model = ex.make_logreg(scaled=True).fit(X_train, y_train)
one = X_val[:1]
probs = model.predict_proba(one)[0]
top3 = np.argsort(probs)[::-1][:3]
print("True person:", y_val[0])
for i in top3:
    print(f"  {model.classes_[i]}: {probs[i]*100:.1f}%")

## 8 · Experiment 2: PCA (fewer features)

**In plain words:** PCA finds the directions in which attempts differ the most and keeps only the top few. It's like photographing a 3D object from its most informative angle.

Question: **how many components do we need before KNN stops getting better?** We also measure speed, because fewer numbers means faster distance calculations.

In [ ]:
pca_table, evr = ex.exp_pca(train, val, best_k)
pca_table.to_csv(TABLES_DIR / "exp2_pca.csv", index=False)
chosen_pca = ex.best_pca(pca_table)
print("Smallest number of components within 1% of the best macro-F1:", chosen_pca)
pca_table.round(4)

In [ ]:
plots.plot_pca_experiment(pca_table, evr, chosen_pca);

👉 **What to notice:**
* *Left:* how many components you need to keep about 90% of the information.
* *Middle:* accuracy usually climbs fast and then flattens; that flat part means extra components add little.
* *Right:* prediction time. Timings on a laptop are noisy, so we take the median of 3 runs.

*Write your observation: how many components would you choose, and why?*

## 9 · Experiment 3: how much enrollment data do we need?

In a real app, nobody will type their password 250 times to sign up. We train with only the first **n** attempts per person and check validation accuracy.

In [ ]:
size_table = ex.exp_training_size(train, val, best_k)
size_table.to_csv(TABLES_DIR / "exp3_training_size.csv", index=False)
plots.plot_training_size(size_table);

In [ ]:
size_table.pivot(index="per_person", columns="model", values="accuracy").round(4)

👉 **What to notice:** how much accuracy you lose with only 5 or 10 attempts per person (realistic for an app sign-up) compared with 250. This is a key **limitation** to discuss.

## 10 · Final test (the "final exam")

Now we fix every choice made on validation (best K, PCA size), retrain on **sessions 1–6**, and evaluate **once** on **sessions 7–8**.

In [ ]:
final, y_test, preds = ex.final_test(trainval, test, best_k, chosen_pca)
final.to_csv(TABLES_DIR / "exp4_final_test.csv", index=False)
final.round(4)

In [ ]:
plots.plot_model_comparison(final);

In [ ]:
best_model_name = final.sort_values("macro_f1", ascending=False).iloc[0]["model"]
print("Best model on test:", best_model_name)
plots.plot_confusion(y_test, preds[best_model_name], best_model_name);

**Reading the confusion matrix:** rows are the true person, columns are who the model guessed. A perfect model would show only the diagonal. Any bright square off the diagonal shows two people being mixed up.

In [ ]:
confused = ex.most_confused_pairs(y_test, preds[best_model_name])
confused.to_csv(TABLES_DIR / "exp4_most_confused_pairs.csv", index=False)
confused

In [ ]:
# Failure case: compare the rhythms of the most-confused pair
if len(confused):
    a, b = confused.iloc[0]["true"], confused.iloc[0]["predicted"]
    plots.eda_subject_rhythms(df, [a, b]);

👉 **Failure case analysis:** if the two lines above nearly overlap, those two people simply type alike, and no distance-based model can separate them well. Mention this in your report.

## 11 · Experiment 5: the login check (verification)

Identification asks *"who is this?"*. A login asks *"is this really Rahul?"*

**Method (for every person):**
1. Build their profile from sessions 1–6: the average of each feature and their typical spread.
2. **Genuine attempts:** their own sessions 7–8.
3. **Impostor attempts:** the first 5 test attempts of each of the other 50 people.
4. Measure each attempt's distance from the profile. Small = similar.

We compare three ways of measuring distance: **Euclidean** (ruler), **Manhattan** (city blocks), and **scaled Manhattan** (city blocks, with each feature divided by that person's own spread).

**The two kinds of mistakes:**
* **FAR** (False Accept Rate): impostors we let in. *Security failure.*
* **FRR** (False Reject Rate): real users we blocked. *Annoying failure.*

A stricter threshold lowers FAR but raises FRR. The point where they are equal is the **EER** (Equal Error Rate), one number to compare methods. **Lower is better.**

In [ ]:
per_subject, ver_summary, pooled = ex.exp_verification(trainval, test)
per_subject.to_csv(TABLES_DIR / "exp5_verification_per_subject.csv", index=False)
ver_summary.to_csv(TABLES_DIR / "exp5_verification_summary.csv", index=False)
ver_summary.round(4)

In [ ]:
plots.plot_eer_by_detector(per_subject);

In [ ]:
plots.plot_far_frr(pooled["scaled_manhattan"]["genuine"], pooled["scaled_manhattan"]["impostor"]);

👉 **What to notice:**
* *Left:* green (real person) distances should sit mostly left of the dashed line and red (impostor) mostly right. The overlap is where mistakes happen.
* *Right:* sliding the threshold right lets more impostors in (FAR ↑) and blocks fewer real users (FRR ↓). A bank would pick a strict point; a phone unlock a lenient one.

In [ ]:
# Who is hardest to protect? (highest EER with the scaled Manhattan detector)
sm = per_subject[per_subject["detector"] == "scaled_manhattan"].sort_values("eer", ascending=False)
sm.head(5).round(4)

## 12 · Optional: test on your group's own typing

In the KeyPrint app, click **Download group data (CSV)** and save the file as `data/keyprint_group_data.csv`. This cell then checks whether the models work on **new people** typing on **your** keyboards. Each person needs at least 10 attempts.

In [ ]:
group_file = ROOT / "data" / "keyprint_group_data.csv"
if group_file.exists():
    g = pd.read_csv(group_file)
    g["subject"] = g["subject"].astype(str)
    g = g.sort_values(["subject", "rep"])
    counts = g.groupby("subject").size()
    g = g[g["subject"].isin(counts[counts >= 10].index)]
    if g["subject"].nunique() >= 2:
        # first 70% of each person's attempts for training, last 30% for testing
        g["order"] = g.groupby("subject").cumcount()
        g["n"] = g.groupby("subject")["rep"].transform("size")
        g_train, g_test = g[g["order"] < 0.7 * g["n"]], g[g["order"] >= 0.7 * g["n"]]
        Xa, ya = X_y(g_train)
        Xb, yb = X_y(g_test)
        rows = []
        for name, m in [("KNN K=3, scaled", ex.make_knn(3, True)), ("Logistic regression, scaled", ex.make_logreg(True))]:
            res, _ = ex.fit_and_score(m, Xa, ya, Xb, yb)
            rows.append({"model": name, **res})
        print(f"{g['subject'].nunique()} group members, {len(g_train)} train / {len(g_test)} test attempts")
        display(pd.DataFrame(rows).round(4))
    else:
        print("Need at least 2 group members with 10+ attempts each.")
else:
    print("No group data yet. Export it from the KeyPrint app to", group_file)

## 13 · Save the environment (reproducibility)

In [ ]:
env = ex.save_environment()
env

## 14 · Analysis and conclusion (fill in with YOUR numbers)

Use the tables above. Never copy numbers from anywhere else.

**What changed and why**
* Scaling changed KNN validation accuracy from ___% to ___%, because …
* The best K was ___. Smaller K was worse/better because …
* PCA with ___ components kept ___% of the variance and reached ___% accuracy, versus ___% with all 31 features.
* With only 5 attempts per person, accuracy was ___%; with 250 it was ___%.

**Final test (sessions 7–8)**
* Baseline (KNN K=1, no scaling): accuracy ___%, macro-F1 ___%
* Best model: ________ with accuracy ___%, macro-F1 ___%

**Login check**
* Mean EER: Euclidean ___%, Manhattan ___%, scaled Manhattan ___%. The best was ___ because …
* The hardest person to protect had an EER of ___%.

**Trade-offs**
* KNN needs no training but must compare against every stored attempt; logistic regression trains slower but predicts much faster.
* A stricter login threshold → fewer impostors (FAR) but more blocked users (FRR).

**Failure cases and limitations**
* The most-confused pair was ___ and ___: their rhythms nearly overlap.
* One fixed password; lab keyboard; volunteers were very practised after 400 repetitions; real users will be noisier.

**Conclusion (2–3 sentences):** …